In [1]:
!python --version  

Python 3.13.12


In [2]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

# avoid warnings for old claude model use
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
# Initialize Antropic Chat client
from anthropic import Anthropic
client = Anthropic()

In [5]:
# Helper Chat Functions


## Function to Append User messasge
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


## Function to Append Assistant messasge
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


## Main Function for chat
def get_response(messages, stop_sequences=None, system=None):

    model = "claude-haiku-4-5"  # you can change the model according to your usecase
    params = {
        "model": model,
        "max_tokens": 5000,
        "messages": messages,
    }
    if system:
        params["system"] = system    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message


def get_chat_content(message):
    return message.content[0].text

In [6]:
# def getStructureOutput(mesg, data_prefix="```json", stop_sequences=None):
#     messages = []

#     # User's request > what user wants in specific format
#     add_user_message(messages, mesg)

#     # start the content and make the AI generate the remaining part.
#     add_assistant_message(messages, data_prefix)

#     response = get_response(messages=messages, stop_sequences=stop_sequences)
#     return get_chat_content(response)

In [7]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = get_chat_content(get_response(messages))
    return output


def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [8]:
import json

with open("./json_db/dataset.json","r") as db:
    results = run_eval(json.load(db))

In [17]:
# With less temprature we will likely get the almost same responses each time

print(json.dumps(results,indent=2))
# response = getStructureOutput(
#     "Generate a very short event bridge rule as json",
#     stop_sequences=["```"],
# )


[
  {
    "output": "# Python Function to Start an EC2 Instance using boto3\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef start_ec2_instance(instance_id, region_name='us-east-1'):\n    \"\"\"\n    Start an Amazon EC2 instance given an instance ID.\n    \n    Args:\n        instance_id (str): The ID of the EC2 instance to start (e.g., 'i-0123456789abcdef0')\n        region_name (str): AWS region name (default: 'us-east-1')\n    \n    Returns:\n        dict: Response containing instance information\n        \n    Raises:\n        ClientError: If the instance doesn't exist or other AWS errors occur\n    \"\"\"\n    try:\n        # Create an EC2 client\n        ec2_client = boto3.client('ec2', region_name=region_name)\n        \n        # Start the instance\n        response = ec2_client.start_instances(InstanceIds=[instance_id])\n        \n        # Print success message\n        print(f\"Successfully started EC2 instance: {instance_id}\")\n        print(f